In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns
import sklearn.metrics
import sklearn.calibration
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import glob, os, unicodedata
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, top_k_accuracy_score, f1_score


In [2]:
def normalize(name):
    """Normalize cancer type names by handling Unicode characters and punctuation."""
    # Normalize unicode (e.g., replace en dash/em dash with hyphen)
    name = unicodedata.normalize('NFKD', name)
    name = name.replace('‑', '-')  # non-breaking hyphen
    name = name.replace('‐', '-')  # hyphen
    name = name.replace('–', '-')  # en dash
    name = name.replace('—', '-')  # em dash
    name = name.replace(',', '')  # remove commas
    # name = name.strip().lower()  # lowercase for uniformity
    return name


In [3]:
"""Load and prepare the genomic data for analysis."""
# Define paths
gdd_path = "/data1/morrisq/yuj13/llm_genomics/genie_output/GDD-ENS/filtered_batch_msk"
o3_mini_path = "/data1/morrisq/yuj13/llm_genomics/genie_output/gpt-5/filtered_batch_msk" #gpt-4o, gpt-5, medgemma
medgemma_path = "/data1/morrisq/yuj13/llm_genomics/genie_output/medgemma/filtered_batch_msk"

# Load validation set
val_set = pd.read_csv("/data1/morrisq/yuj13/llm_genomics/genie_output/o3-mini/output_nosite_extended_o3-mini.csv")
val_set["SAMPLE_ID"] = "GENIE-MSK-" + val_set["SAMPLE_ID"].astype(str)

# Load clinical data and mapping
data_clinical_sample = pd.read_csv('/data1/morrisq/yuj13/llm_genomics/genie_data_v17.0/data_clinical_sample.txt',sep='\t',comment='#')
mapping_df = pd.read_table("/data1/morrisq/yuj13/llm_genomics/GDD_ENS/data/tumor_type_final.txt")
mapping_df_unique = mapping_df[['CANCER_TYPE', 'Cancer_Type']].drop_duplicates()
mapping = dict(zip(mapping_df_unique['Cancer_Type'], mapping_df_unique['CANCER_TYPE']))

# Load MSK data
if len(glob.glob(os.path.join(gdd_path, "*.csv"))) > 0:
    val_cancertypes_gdd_msk = pd.concat((pd.read_csv(f) for f in glob.glob(os.path.join(gdd_path, "*.csv"))), ignore_index=True)
    val_cancertypes_gdd_msk = val_cancertypes_gdd_msk[val_cancertypes_gdd_msk["SAMPLE_ID"].isin(val_set["SAMPLE_ID"])]

if len(glob.glob(os.path.join(o3_mini_path, "*.csv"))) > 0:
    val_cancertypes_msk = pd.concat((pd.read_csv(f) for f in glob.glob(os.path.join(o3_mini_path, "*.csv"))), ignore_index=True)
    val_cancertypes_msk = val_cancertypes_msk[val_cancertypes_msk["SAMPLE_ID"].isin(val_set["SAMPLE_ID"])]

if len(glob.glob(os.path.join(medgemma_path, "*.csv"))) > 0:
    print(len(glob.glob(os.path.join(medgemma_path, "*.csv"))))
    val_cancertypes_gemma_msk = pd.concat((pd.read_csv(f) for f in glob.glob(os.path.join(medgemma_path, "*.csv"))), ignore_index=True)
    val_cancertypes_gemma_msk = val_cancertypes_gemma_msk[val_cancertypes_gemma_msk["SAMPLE_ID"].isin(val_set["SAMPLE_ID"])]

# Load non-MSK data
gdd_path_non_msk = "/data1/morrisq/yuj13/llm_genomics/genie_output/GDD-ENS/filtered_batch_non_msk"
o3_mini_path_non_msk = "/data1/morrisq/yuj13/llm_genomics/genie_output/gpt-5/filtered_batch_non_msk"
medgemma_path_non_msk = "/data1/morrisq/yuj13/llm_genomics/genie_output/medgemma/filtered_batch_non_msk"

val_cancertypes_gdd = pd.concat((pd.read_csv(f) for f in glob.glob(os.path.join(gdd_path_non_msk, "*.csv"))), ignore_index=True)
val_cancertypes = pd.concat((pd.read_csv(f) for f in glob.glob(os.path.join(o3_mini_path_non_msk, "*.csv"))), ignore_index=True)
val_cancertypes_gemma = pd.concat((pd.read_csv(f) for f in glob.glob(os.path.join(medgemma_path_non_msk, "*.csv"))), ignore_index=True)

# Combine MSK and non-MSK data
val_cancertypes_gdd = pd.concat([val_cancertypes_gdd, val_cancertypes_gdd_msk], ignore_index=True).drop_duplicates(subset="SAMPLE_ID")
val_cancertypes = pd.concat([val_cancertypes, val_cancertypes_msk], ignore_index=True).drop_duplicates(subset="SAMPLE_ID")
val_cancertypes_gemma = pd.concat([val_cancertypes_gemma, val_cancertypes_gemma_msk], ignore_index=True).drop_duplicates(subset="SAMPLE_ID")

    
# ------Process GDD data with cancer type mapping-------
print("------------------GDD-ENS-------------------")
val_cancertypes_gdd = val_cancertypes_gdd.rename(columns={'Cancer_Type': 'ground_truth'})
ground_truth_cancers = val_cancertypes_gdd.ground_truth.unique()
gdd_pred_cancers = val_cancertypes_gdd.Pred1.unique()

# Merge with clinical data
val_cancertypes_gdd = val_cancertypes_gdd.merge(
    data_clinical_sample[['SAMPLE_ID', 'CANCER_TYPE_DETAILED', 'SEQ_ASSAY_ID']],
    on='SAMPLE_ID',
    how='left'
)

# Map predictions using mapping_df for Pred1
val_cancertypes_gdd = val_cancertypes_gdd.merge(
    mapping_df,
    left_on=['CANCER_TYPE_DETAILED', 'Pred1'],
    right_on=['CANCER_TYPE_DETAILED', 'Cancer_Type'],
    how='left'
)
val_cancertypes_gdd = val_cancertypes_gdd.rename(columns={'CANCER_TYPE': 'prediction1'})
val_cancertypes_gdd['prediction1'] = (
    val_cancertypes_gdd['prediction1']
    .fillna(val_cancertypes_gdd['Pred1'].map(mapping))
    .fillna('Unknown')
)

# Map predictions using mapping_df for Pred2
val_cancertypes_gdd = val_cancertypes_gdd.merge(
    mapping_df,
    left_on=['CANCER_TYPE_DETAILED', 'Pred2'],
    right_on=['CANCER_TYPE_DETAILED', 'Cancer_Type'],
    how='left'
)
val_cancertypes_gdd = val_cancertypes_gdd.rename(columns={'CANCER_TYPE': 'prediction2'})
val_cancertypes_gdd['prediction2'] = (
    val_cancertypes_gdd['prediction2']
    .fillna(val_cancertypes_gdd['Pred2'].map(mapping))
    .fillna('Unknown')
)
val_cancertypes_gdd = val_cancertypes_gdd.rename(columns={'Conf1': 'probs'})
val_cancertypes_gdd = val_cancertypes_gdd.rename(columns={'Conf2': 'probs2'})

# Mapping to numbers
# Get unique values from ground_truth column and sort them alphabetically
# Get all unique values from ground_truth, prediction1, and prediction2
all_unique_values = set(val_cancertypes_gdd['ground_truth'].unique()) | \
                   set(val_cancertypes_gdd['prediction1'].unique()) | \
                   set(val_cancertypes_gdd['prediction2'].unique())

# Sort to ensure consistent ordering
all_unique_values = sorted(all_unique_values)

# Create comprehensive mapping
ground_truth_mapping = {value: idx for idx, value in enumerate(all_unique_values)}

# Apply the mapping
val_cancertypes_gdd['true'] = val_cancertypes_gdd['ground_truth'].map(ground_truth_mapping)
val_cancertypes_gdd['pred'] = val_cancertypes_gdd['prediction1'].map(ground_truth_mapping)
val_cancertypes_gdd['pred2'] = val_cancertypes_gdd['prediction2'].map(ground_truth_mapping)

unknown_rows_gdd = val_cancertypes_gdd[val_cancertypes_gdd['prediction1'] == "UNKNOWN"]
print(f"Number of Unknown rows in GDD predictions: {len(unknown_rows_gdd)}")

print("-------------------------------------------------------------")
# ------Process o3-mini data with cancer type mapping-------
print("------------------O3-mini-------------------")

val_cancertypes['prediction1'] = val_cancertypes['prediction1'].fillna('UNKNOWN')
val_cancertypes['prediction2'] = val_cancertypes['prediction2'].fillna('UNKNOWN')
val_cancertypes['prediction1'] = val_cancertypes['prediction1'].apply(normalize)
val_cancertypes['prediction2'] = val_cancertypes['prediction2'].apply(normalize)

# Fix specific cancer type naming inconsistencies
# Define the values to replace
values_to_replace = {'Indeterminate', 'Not Applicable', 'Unclassifiable', 'Undetermined'}

# Assuming val_cancertypes is your DataFrame
# Replace the specified values with 'UNKNOWN' in prediction1 and prediction2 columns
val_cancertypes['prediction1'] = val_cancertypes['prediction1'].replace(values_to_replace, 'UNKNOWN')
val_cancertypes['prediction2'] = val_cancertypes['prediction2'].replace(values_to_replace, 'UNKNOWN')

val_cancertypes.loc[val_cancertypes['prediction1'] == 'Skin Cancer Non-Melanoma', 'prediction1'] = 'Skin Cancer, Non-Melanoma'
val_cancertypes.loc[val_cancertypes['prediction1'] == 'Non-Small-Cell Lung Cancer', 'prediction1'] = 'Non-Small Cell Lung Cancer'
val_cancertypes.loc[val_cancertypes['prediction1'] == 'CNS Cancer (likely meningioma)', 'prediction1'] = 'CNS Cancer'
val_cancertypes.loc[val_cancertypes['prediction1'] == 'Gastrointestinal Stromat Tumor', 'prediction1'] = 'Gastrointestinal Stromal Tumor'
val_cancertypes.loc[val_cancertypes['prediction1'] == 'Gastrointestinal Stromatic Tumor', 'prediction1'] = 'Gastrointestinal Stromal Tumor'
val_cancertypes.loc[val_cancertypes['prediction1'] == 'Gastrointestinal Stromenchymal Tumor', 'prediction1'] = 'Gastrointestinal Stromal Tumor'
other_cancers = set(val_cancertypes.prediction1.unique()) - set(val_cancertypes.ground_truth.unique()) - {'UNKNOWN'}
print(f"Other cancer type in O3-mini prediction1: {other_cancers}")
val_cancertypes['prediction1'] = val_cancertypes['prediction1'].replace(other_cancers, 'OTHERS')

val_cancertypes.loc[val_cancertypes['prediction2'] == 'Skin Cancer Non-Melanoma', 'prediction2'] = 'Skin Cancer, Non-Melanoma'
val_cancertypes.loc[val_cancertypes['prediction2'] == 'Non-Small Cell Lung Cancer', 'prediction2'] = 'Non-Small Cell Lung Cancer'
val_cancertypes.loc[val_cancertypes['prediction2'] == 'Malignant Mesothelioma', 'prediction2'] = 'Mesothelioma'
val_cancertypes.loc[val_cancertypes['prediction2'] == 'Papillary Thyroid Cancer', 'prediction2'] = 'Thyroid Cancer'
val_cancertypes.loc[val_cancertypes['prediction2'] == 'Gastrointestinal Stromen Tumor', 'prediction2'] = 'Gastrointestinal Stromal Tumor'
other_cancers = set(val_cancertypes.prediction2.unique()) - set(val_cancertypes.ground_truth.unique()) - {'UNKNOWN'}
print(f"Other cancer type in O3-mini prediction2: {other_cancers}")
val_cancertypes['prediction2'] = val_cancertypes['prediction2'].replace(other_cancers, 'OTHERS')

# Mapping to numbers
# Get unique values from ground_truth column and sort them alphabetically
unique_ground_truth = sorted(val_cancertypes['ground_truth'].unique())
ground_truth_mapping = {value: idx for idx, value in enumerate(unique_ground_truth)}
val_cancertypes['true'] = val_cancertypes['ground_truth'].map(ground_truth_mapping)
val_cancertypes['pred'] = val_cancertypes['prediction1'].map(ground_truth_mapping)
val_cancertypes['pred2'] = val_cancertypes['prediction2'].map(ground_truth_mapping)

# Mapping to numbers
# Get unique values from ground_truth column and sort them alphabetically
# Get all unique values from ground_truth, prediction1, and prediction2
all_unique_values = set(val_cancertypes['ground_truth'].unique()) | \
                   set(val_cancertypes['prediction1'].unique()) | \
                   set(val_cancertypes['prediction2'].unique())

# Sort to ensure consistent ordering
all_unique_values = sorted(all_unique_values)

# Create comprehensive mapping
ground_truth_mapping = {value: idx for idx, value in enumerate(all_unique_values)}

# Apply the mapping
val_cancertypes['true'] = val_cancertypes['ground_truth'].map(ground_truth_mapping)
val_cancertypes['pred'] = val_cancertypes['prediction1'].map(ground_truth_mapping)
val_cancertypes['pred2'] = val_cancertypes['prediction2'].map(ground_truth_mapping)

val_cancertypes = val_cancertypes.rename(columns={'prob1': 'probs'})
val_cancertypes = val_cancertypes.rename(columns={'prob2': 'probs2'})

val_cancertypes = val_cancertypes.merge(
    data_clinical_sample[['SAMPLE_ID', 'CANCER_TYPE_DETAILED', 'SEQ_ASSAY_ID']],
    on='SAMPLE_ID',
    how='left'
)
unknown_rows_o3_mini = val_cancertypes[val_cancertypes['prediction1'] == "UNKNOWN"]
others_rows_o3_mini = val_cancertypes[val_cancertypes['prediction1'] == "OTHERS"]
print(f"Number of UNKNOWN rows in O3-mini predictions: {len(unknown_rows_o3_mini)}")
print(f"Number of OTHERS rows in O3-mini predictions: {len(others_rows_o3_mini)}")
print("-------------------------------------------------------------")

# ------Process medgemma data with cancer type mapping-------
print("------------------MedGemma-------------------")
val_cancertypes_gemma['prediction1'] = val_cancertypes_gemma['prediction1'].fillna('UNKNOWN')
val_cancertypes_gemma['prediction2'] = val_cancertypes_gemma['prediction2'].fillna('UNKNOWN')
val_cancertypes_gemma['prediction1'] = val_cancertypes_gemma['prediction1'].apply(normalize)
val_cancertypes_gemma['prediction2'] = val_cancertypes_gemma['prediction2'].apply(normalize)

# Fix specific cancer type naming inconsistencies
# Define the values to replace
values_to_replace = {'UNKNOWN', 'Unknown'}

# Assuming val_cancertypes is your DataFrame
# Replace the specified values with 'UNKNOWN' in prediction1 and prediction2 columns
val_cancertypes_gemma['prediction1'] = val_cancertypes_gemma['prediction1'].replace(values_to_replace, 'UNKNOWN')
val_cancertypes_gemma['prediction2'] = val_cancertypes_gemma['prediction2'].replace(values_to_replace, 'UNKNOWN')

val_cancertypes_gemma.loc[val_cancertypes_gemma['prediction1'] == 'Skin Cancer Non-Melanoma', 'prediction1'] = 'Skin Cancer, Non-Melanoma'
val_cancertypes_gemma.loc[val_cancertypes_gemma['prediction1'] == 'Clear Cell Renal Cell Carcinoma', 'prediction1'] = 'Renal Cell Carcinoma'
val_cancertypes_gemma.loc[val_cancertypes_gemma['prediction1'] == 'Papillary Cell Renal Cell Carcinoma', 'prediction1'] = 'Renal Cell Carcinoma'
val_cancertypes_gemma.loc[val_cancertypes_gemma['prediction1'] == 'Urothelial Cancer (Bladder Cancer)', 'prediction1'] = 'Bladder Cancer'
other_cancers = set(val_cancertypes_gemma.prediction1.unique()) - set(val_cancertypes_gemma.ground_truth.unique()) - {'UNKNOWN'}
print(f"Other cancer type in MedGemma prediction1: {other_cancers}")
val_cancertypes_gemma['prediction1'] = val_cancertypes_gemma['prediction1'].replace(other_cancers, 'OTHERS')

val_cancertypes_gemma.loc[val_cancertypes_gemma['prediction2'] == 'Skin Cancer Non-Melanoma', 'prediction2'] = 'Skin Cancer, Non-Melanoma'
val_cancertypes_gemma.loc[val_cancertypes_gemma['prediction2'] == 'Clear Cell Renal Cell Carcinoma', 'prediction2'] = 'Renal Cell Carcinoma'
val_cancertypes_gemma.loc[val_cancertypes_gemma['prediction2'] == 'Papillary Cell Renal Cell Carcinoma', 'prediction2'] = 'Renal Cell Carcinoma'
other_cancers = set(val_cancertypes_gemma.prediction2.unique()) - set(val_cancertypes_gemma.ground_truth.unique()) - {'UNKNOWN'}
print(f"Other cancer type in MedGemma prediction2: {other_cancers}")
val_cancertypes_gemma['prediction2'] = val_cancertypes_gemma['prediction2'].replace(other_cancers, 'OTHERS')

# Get all unique values from ground_truth, prediction1, and prediction2
all_unique_values = set(val_cancertypes_gemma['ground_truth'].unique()) | \
                   set(val_cancertypes_gemma['prediction1'].unique()) | \
                   set(val_cancertypes_gemma['prediction2'].unique())

# Sort to ensure consistent ordering
all_unique_values = sorted(all_unique_values)

# Create comprehensive mapping
ground_truth_mapping = {value: idx for idx, value in enumerate(all_unique_values)}

# Apply the mapping
val_cancertypes_gemma['true'] = val_cancertypes_gemma['ground_truth'].map(ground_truth_mapping)
val_cancertypes_gemma['pred'] = val_cancertypes_gemma['prediction1'].map(ground_truth_mapping)
val_cancertypes_gemma['pred2'] = val_cancertypes_gemma['prediction2'].map(ground_truth_mapping)

val_cancertypes_gemma = val_cancertypes_gemma.rename(columns={'prob1': 'probs'})
val_cancertypes_gemma = val_cancertypes_gemma.rename(columns={'prob2': 'probs2'})
val_cancertypes_gemma = val_cancertypes_gemma.merge(
    data_clinical_sample[['SAMPLE_ID', 'CANCER_TYPE_DETAILED', 'SEQ_ASSAY_ID']],
    on='SAMPLE_ID',
    how='left'
)
unknown_rows_gemma = val_cancertypes_gemma[val_cancertypes_gemma['prediction1'] == "UNKNOWN"]
others_rows_gemma = val_cancertypes_gemma[val_cancertypes_gemma['prediction1'] == "OTHERS"]
print(f"Number of UNKNOWN rows in MedGemma predictions: {len(unknown_rows_gemma)}")
print(f"Number of OTHERS rows in MedGemma predictions: {len(others_rows_gemma)}")
print("-------------------------------------------------------------")


1
------------------GDD-ENS-------------------
Number of Unknown rows in GDD predictions: 0
-------------------------------------------------------------
------------------O3-mini-------------------
Other cancer type in O3-mini prediction1: {'Gastrointestinal Stromor Tumor', 'Non-Small C Lung Cancer', 'Non-Small Core Lung Cancer', 'Esophagogastrric Cancer'}
Other cancer type in O3-mini prediction2: {'Gastrointestinal Stromor Tumor'}
Number of UNKNOWN rows in O3-mini predictions: 0
Number of OTHERS rows in O3-mini predictions: 4
-------------------------------------------------------------
------------------MedGemma-------------------
Other cancer type in MedGemma prediction1: {'Acute Myeloid Leukemia', 'Lung Cancer', 'Cholangiocarcinoma', 'Testicular Cancer', 'Pituitary Adenoma', 'Gastric Cancer', 'Hematopoietic and Lymphoid Cancer', 'Hematopoietic and Lymphoid Tissue Cancer', 'B Cell Lymphoma', 'Medulloblastoma', 'Urothelial Cancer', 'Neuroendocrine Tumor', 'Neuroblastoma', 'Glioblast

In [4]:
# Load the assay info table
assay_df = pd.read_table("/data1/morrisq/yuj13/llm_genomics/genie_data_v17.0/assay_information.txt")

# Unique SEQ_ASSAY_ID and CENTER pairs
assay_df_unique = assay_df[['SEQ_ASSAY_ID', 'CENTER']].drop_duplicates()
seq_assay_to_center = dict(zip(assay_df_unique['SEQ_ASSAY_ID'], assay_df_unique['CENTER']))
val_cancertypes_gemma['CENTER'] = val_cancertypes_gemma['SEQ_ASSAY_ID'].map(seq_assay_to_center)
val_cancertypes_gdd['CENTER'] = val_cancertypes_gemma['SEQ_ASSAY_ID'].map(seq_assay_to_center)
val_cancertypes['CENTER'] = val_cancertypes_gemma['SEQ_ASSAY_ID'].map(seq_assay_to_center)


In [5]:
from typing import List, Tuple
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score

REQUIRED_COLS = {"SAMPLE_ID", "prediction1", "prediction2", "ground_truth", "CENTER"}

def stratified_center_performance(dfs: List[Tuple[str, pd.DataFrame]]) -> pd.DataFrame:
    """
    Parameters
    ----------
    dfs : list of (method_name, DataFrame)
        Each DataFrame must have columns:
        SAMPLE_ID, prediction1, prediction2, ground_truth, CENTER

    Returns
    -------
    pd.DataFrame
        Columns:
        CENTER, Method, N, Accuracy, Top2_Accuracy, Weighted_Precision, Weighted_Recall, Weighted_F1, CENTER_N
    """
    rows = []

    for method, df in dfs:
        missing = REQUIRED_COLS - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns for method '{method}': {sorted(missing)}")

        # Keep rows with valid center & ground truth
        dd = df.dropna(subset=["CENTER", "ground_truth"]).copy()

        for center, g in dd.groupby("CENTER", dropna=False):
            y_true = g["ground_truth"]
            y_pred1 = g["prediction1"]
            y_pred2 = g["prediction2"]

            N = len(g)
            if N == 0:
                continue

            # Metrics
            acc = accuracy_score(y_true, y_pred1)
            top2_acc = ((y_pred1 == y_true) | (y_pred2 == y_true)).mean()

            w_precision = precision_score(
                y_true, y_pred1, average="weighted", zero_division=0
            )
            w_recall = recall_score(
                y_true, y_pred1, average="weighted", zero_division=0
            )
            w_f1 = f1_score(
                y_true, y_pred1, average="weighted", zero_division=0
            )
            rows.append(
                {
                    "CENTER": center,
                    "Method": method,
                    "N": N,
                    "Accuracy": float(acc),
                    "Top2_Accuracy": float(top2_acc),
                    "Weighted_Precision": float(w_precision),
                    "Weighted_Recall": float(w_recall),
                    "Weighted_F1": float(w_f1),
                    "CENTER_N": f"{center} (N={N})",
                }
            )

    out = pd.DataFrame(rows)[
        [
            "CENTER",
            "Method",
            "N",
            "Accuracy",
            "Top2_Accuracy",
            "Weighted_Precision",
            "Weighted_Recall",
            "Weighted_F1",
            "CENTER_N",
        ]
    ].sort_values(["CENTER", "Method"], kind="stable")

    return out.reset_index(drop=True)


result = stratified_center_performance([
    # ("GDD-ENS", val_cancertypes_gdd),
    # ("medgemma", val_cancertypes_gemma),
    ("gpt-5", val_cancertypes),
])


In [6]:
result

,CENTER,Method,N,Accuracy,Top2_Accuracy,Weighted_Precision,Weighted_Recall,Weighted_F1,CENTER_N
0,CHOP,gpt-5,848,0.485849,0.603774,0.524804,0.485849,0.488504,CHOP (N=848)
1,COLU,gpt-5,5224,0.494832,0.600115,0.566493,0.494832,0.509640,COLU (N=5224)
2,CRUK,gpt-5,1997,0.451678,0.569855,0.514255,0.451678,0.462671,CRUK (N=1997)
3,DFCI,gpt-5,33479,0.478867,0.595926,0.522233,0.478867,0.486785,DFCI (N=33479)
4,DUKE,gpt-5,3340,0.453293,0.578144,0.503288,0.453293,0.460183,DUKE (N=3340)
5,GRCC,gpt-5,1068,0.471910,0.595506,0.544052,0.471910,0.487282,GRCC (N=1068)
6,JHU,gpt-5,5307,0.493499,0.608442,0.538551,0.493499,0.502403,JHU (N=5307)
7,MDA,gpt-5,2404,0.507072,0.623128,0.568331,0.507072,0.526642,MDA (N=2404)
8,MSK,gpt-5,6122,0.581999,0.695034,0.626754,0.581999,0.587509,MSK (N=6122)
9,NKI,gpt-5,3189,0.498589,0.608341,0.532633,0.498589,0.502851,NKI (N=3189)


In [7]:
result.sort_values(by=["Method", "CENTER"]).to_csv("/data1/morrisq/yuj13/llm_genomics/genie_output/performance_stratified_by_center_.csv",index=False)
